# Systems to Blend Different Feed Streams


In [ ]:
from pathlib import Path

import casadi as cas
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from cas_models.continuous_time.models import StateSpaceModelCT
from cas_models.discrete_time.models import StateSpaceModelDTFromCTRK4
from cas_models.discrete_time.simulate import (
    make_n_step_simulation_function_from_model,
)
from cas_models.transformations import connect_systems
from python_pid import PIDController

from bounded_random_walk import sample_bounded_random_walk
from feed_conc_ctrl.models import FlowMixerCT, MixingTankModelCT
from feed_conc_ctrl.plot_utils import make_tsplots


In [ ]:
plot_dir = Path("./plots")
plot_dir.mkdir(exist_ok=True)


## Generate Bounded Random Walk (BRW) Sequences


In [ ]:
seed = 100
rng = np.random.default_rng(seed)

# Noise std. dev.
sd_e = 5.0

# Bounded random walk parameters
r1 = -40.0  # When x = r1, bias = +1 (pushes up)
r2 = 40.0  # When x = r2, bias = -1 (pushes down)
a1 = 0.2  # aggressiveness of lower bound
a2 = 0.2  # aggressiveness of upper bound

# Number of random walks to generate
n_walks = 3

nT = 600
bounded_random_walks = sample_bounded_random_walk(
    sd_e, r1, r2, a1, a2, nT, n_walks=3, rng=rng
)
assert bounded_random_walks.shape == (nT, n_walks)

# Nominal input value
u_nop = 50.0

# Time vector
Ts = 1.0
t = Ts * np.arange(nT)


In [ ]:
# Only plot the first t_stop minutes of each BRW
t_stop = 600.0
nT_plot = int(np.floor(t_stop / Ts))

n_plots = min(5, n_walks)

marker = ""

fig, axes = plt.subplots(n_plots, 1, sharex=True, figsize=(7, 1 + 1.5*n_plots))

for i, ax in enumerate(axes):
    u = bounded_random_walks[:, i]
    ax.plot(t[:nT_plot], u_nop + u[:nT_plot], marker=marker)
    ax.axhline(u_nop + r1, linestyle='--', color='grey', label='min')
    ax.axhline(u_nop + r2, linestyle='--', color='grey', label='max')
    ax.set_ylim([u_nop + 1.3 * r1, u_nop + 1.3 * r2])
    ax.set_ylabel("%")
    ax.grid()
    ax.set_title(f"Bounded Random Walk {i+1:d}")

ax.set_xlabel("Time (mins)")
plt.tight_layout()
filename = "bounded_random_walks.png"
plt.savefig(plot_dir / filename, dpi=300)
plt.show()


In [ ]:
# Convert two BRWs into independent concentration disturbances
c_bounds =  [(20, 35), (65, 80)]
c_nom = [np.mean(c_bounds[0]), np.mean(c_bounds[1])]
# Nominal concentrations of input streams 1 and 2
print(f"c_nom: {c_nom}")

c_1 = c_nom[0] + bounded_random_walks[:, 0] * (np.diff(c_bounds[0])) / 50.0
c_2 = c_nom[1] + bounded_random_walks[:, 1] * (np.diff(c_bounds[1])) / 50.0

assert c_1.shape == (nT,)
assert c_2.shape == (nT,)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.5))

ax.plot(t[:nT_plot], c_1[:nT_plot], marker=marker, label='c_1')
ax.plot(t[:nT_plot], c_2[:nT_plot], marker=marker, label='c_2')
for i in range(2):
    ax.fill_between(
        t[:nT_plot],
        c_bounds[i][0],
        c_bounds[i][1],
        color=f"C{i}",
        alpha=0.1,
        label=f"c_{i+1} bounds",
    )

# Add target concentration line
tank_conc_out_sp = 50.0  # setpoint for concentration
ax.axhline(tank_conc_out_sp, linestyle='--', color='C4', label='Setpoint')

ax.set_ylim([0, 100])
ax.set_xlabel("Time (mins)")
ax.set_ylabel("Composition (%)")
ax.grid()
ax.legend(loc='upper right')
#ax.set_title(f"Bounded Random Walks")

plt.tight_layout()
filename = "blending_bounded_random_walks.png"
plt.savefig(plot_dir / filename, dpi=300)
plt.show()


## Construct Mixer and Tank Dynamic System


In [ ]:
def print_sys_dimensions(sys):
    print(sys.name, f"({sys.ny}x{sys.nu})")
    for attr_name in ["input_names", "state_names", "output_names"]:
        print(f"{attr_name:>15s}: {getattr(sys, attr_name)}")


In [ ]:
# Mixing tank dimensions
H = 2   # Height [m]
A = 5  # Cross-sectional area [m^2]
D = np.sqrt(4 * A / np.pi)  # Diameter [m]
total_volume = A * H
flow_rate = 1.0  # m³/min
tau_h = total_volume / flow_rate

print(f"Tank diameter: {D:.2f} m")
print(f"Tank volume: {total_volume:.1f} m³")
print(f"Tank residence time: {tau_h:.1f} min")

mixing_tank_model = MixingTankModelCT(D=D, name="mixing_tank")
print_sys_dimensions(mixing_tank_model)


In [ ]:
# Simple 2-input flow mixer
mixer_model = FlowMixerCT(2, name="mixer")
print_sys_dimensions(mixer_model)


In [ ]:
connections = {
    "mixing_tank_v_dot_in": "mixer_v_dot_out",
    "mixing_tank_conc_in": "mixer_conc_out",
}

model_class = StateSpaceModelCT
mixer_tank_system = connect_systems(
    [mixer_model, mixing_tank_model],
    connections,
    model_class,
    name="mixer_tank_system",
    verbose_names=True,
)
print_sys_dimensions(mixer_tank_system)


In [ ]:
# Convert to discrete-time model
dt = Ts
mixer_tank_system_dt = StateSpaceModelDTFromCTRK4(mixer_tank_system, dt)
print_sys_dimensions(mixer_tank_system_dt)


In [ ]:
simulate = make_n_step_simulation_function_from_model(mixer_tank_system_dt, nT)
simulate


In [ ]:
# Input sequence
mixer_v_dot_in_1 = np.full(nT, 0.5)
mixer_v_dot_in_1[t >= 300] = 0.75 * mixer_v_dot_in_1[t >= 300]
mixer_conc_in_1 = c_1  # np.full(nT, c_nom[0])
mixer_v_dot_in_2 = np.full(nT, 0.5)
mixer_v_dot_in_2[t >= 300] = 0.75 * mixer_v_dot_in_2[t >= 300]
mixer_conc_in_2 = c_2  # np.full(nT, c_nom[1])
mixing_tank_v_dot_out = mixer_v_dot_in_1 + mixer_v_dot_in_2

U = np.stack([
    mixer_v_dot_in_1,
    mixer_conc_in_1,
    mixer_v_dot_in_2,
    mixer_conc_in_2,
    mixing_tank_v_dot_out
]).T

assert U.shape == (nT, mixer_tank_system_dt.nu)


In [ ]:
# Initial condition
tank_L = H  # Start full
tank_m = 500.0
x0 = np.array([tank_L, tank_m])

# Simulation output time vector
t_eval = Ts * np.arange(nT + 1)

X, Y = simulate(t_eval, U, x0)
assert X.shape == (nT + 1, mixer_tank_system_dt.n)
assert Y.shape == (nT + 1, mixer_tank_system_dt.ny)

sim_results = pd.concat(
    [
        pd.DataFrame(t_eval, columns=["time"]),
        pd.DataFrame(U, columns=mixer_tank_system_dt.input_names),
        pd.DataFrame(Y, columns=mixer_tank_system_dt.output_names),
    ],
    axis=1,
)
sim_results = sim_results.set_index("time")
sim_results.index.name = "Time (mins)"
sim_results


In [ ]:
flow_units = "$m^3/min$"
conc_units = "%"
level_units = "m"
mass_units = "kg"

units = {
    "mixer_v_dot_in_1": flow_units,
    "mixer_conc_in_1": conc_units,
    "mixer_v_dot_in_2": flow_units,
    "mixer_conc_in_2": conc_units,
    "mixing_tank_v_dot_out": flow_units,
    "mixer_v_dot_out": flow_units,
    "mixer_conc_out": conc_units,
    "mixing_tank_L": level_units,
    "mixing_tank_m": mass_units,
    "mixing_tank_conc_out": conc_units
}

# Define plot structure
plot_info = {
    # "Mixing Tank Level": {
    #     "Mixing Tank": {"var_name": "mixing_tank_L"},
    # },
    "Flow Rates": {
        "In 1": {"var_name": "mixer_v_dot_in_1", "color": "C0"},
        "In 2": {"var_name": "mixer_v_dot_in_2", "color": "C1"},
        "Out": {"var_name": "mixing_tank_v_dot_out", "color": "C4"},
    },
    "Compositions": {
        "In 1": {"var_name": "mixer_conc_in_1", "color": "C0"},
        "In 2": {"var_name": "mixer_conc_in_2", "color": "C1"},
        "Out": {"var_name": "mixing_tank_conc_out", "color": "C4"},
    }
}

# Create plots
fig, axes = make_tsplots(
    sim_results,
    plot_info,
    units=units,
)

# Add target concentration line
tank_conc_out_sp = 50.0  # setpoint for concentration
axes[1].axhline(tank_conc_out_sp, linestyle="--", color="C4", label="Setpoint")

plt.tight_layout()
filename = "blending_sim_no_ctrl.png"
plt.savefig(plot_dir / filename, dpi=300)
plt.show()


In [ ]:
rmse_tracking = float(
    np.sqrt(
        np.mean((tank_conc_out_sp - sim_results["mixing_tank_conc_out"]) ** 2)
    )
)
rmse_tracking


## Simulate with For Loop


In [ ]:
# Initial condition
mixing_tank_L = H  # Start full
mixing_tank_m = 500.0
x0 = np.array([mixing_tank_L, mixing_tank_m])

# Simulation output time vector
t_eval = Ts * np.arange(nT + 1)
assert t_eval.shape == (nT + 1, )

model = mixer_tank_system_dt

# No model parameters in this case
params = {}

# Simulation loop
X = [cas.DM(x0).T]
Y = []
xk = x0
tk = t_eval[0]
for k in range(nT):
    tkp1 = t_eval[k + 1]
    uk = cas.DM(U[k, :]).T
    xkp1 = model.F(tk, xk, uk, *params.values())
    yk = model.H(tk, xk, uk, *params.values())
    X.append(xkp1.T)
    Y.append(yk.T)
    tk = tkp1
    xk = xkp1

yk = model.H(tk, xk, uk, *params.values())
Y.append(yk.T)
X = np.array(cas.vcat(X))
Y = np.array(cas.vcat(Y))

assert X.shape == (nT + 1, mixer_tank_system_dt.n)
assert Y.shape == (nT + 1, mixer_tank_system_dt.ny)

sim_results = pd.concat(
    [
        pd.DataFrame(t_eval, columns=["time"]),
        pd.DataFrame(U, columns=mixer_tank_system_dt.input_names),
        pd.DataFrame(Y, columns=mixer_tank_system_dt.output_names),
    ],
    axis=1,
)
sim_results = sim_results.set_index("time")
sim_results.index.name = "Time (mins)"
sim_results


In [ ]:
# Create plots
fig, axes = make_tsplots(
    sim_results,
    plot_info,
    units=units,
)

# Add target concentration line
tank_conc_out_sp = 50.0  # setpoint for concentration
axes[-1].axhline(
    tank_conc_out_sp, linestyle="--", color="C4", label="Setpoint"
)

plt.tight_layout()
plt.show()


In [ ]:
rmse_tracking = float(
    np.sqrt(
        np.mean((tank_conc_out_sp - sim_results["mixing_tank_conc_out"]) ** 2)
    )
)
rmse_tracking


## Ratio Control


In [ ]:
# PI regulator for the ratio trim controller
kp = 1.0
ki = 1.0
kd = 0.0
TfTs = 5.0
ctrl = PIDController(kp, ki, kd, TfTs=TfTs)

# Test
assert np.isclose(ctrl(0.0, 0.0, Tx=Ts), 0.0)
u_actuals = []
for y in [0.0, 1.0, 1.0, 1.0, -5.0, -5.0, -5.0]:
    u = ctrl(0.0, y, Tx=Ts)
    u_actuals.append(float(u))

assert np.allclose(
    u_actuals, 
    [ 0.    , -0.035 , -0.1406, -0.3229, -0.3731, -0.0769,  0.6061], 
    atol=0.001
)


In [ ]:
# Initial condition
mixing_tank_L = H  # Start full
mixing_tank_m = 500.0
x0 = np.array([mixing_tank_L, mixing_tank_m])

# Simulation output time vector
t_eval = Ts * np.arange(nT + 1)
assert t_eval.shape == (nT + 1, )

model = mixer_tank_system_dt

# Flow rate disturbance
mixing_tank_v_dot_out = np.full(nT+1, 1.0)
mixing_tank_v_dot_out[t_eval >= 300] = 0.75

# Concentration disturbances
# c_1 = np.full(nT, c_nom[0])
# c_1[t >= 200] = 20.0
# c_2 = np.full(nT, c_nom[1])
# c_2[t >= 400] = 65.0

# Measurement noise
sd_e = 0.5
meas_error = np.random.normal(0, sd_e, size=nT+1)

# No model parameters in this case
params = {}

# PI regulator for ratio trim controller
kp = 0.05
ki = 0.001
kd = 0.0
TfTs = 10.0
umin = -0.999
umax = 0.999
ctrl = PIDController(kp, ki, kd, TfTs=TfTs)

# Normal operating point of ratio: v_dot_in_2 / v_dot_in_1
R_nop = 1.0

# Controller state
u = 0.0
# Simulation loop
U = []
X = [cas.DM(x0).T]
Y = []
xk = x0
tk = t_eval[0]
for k in range(nT):
    tkp1 = t_eval[k + 1]

    # Disturbance inputs
    mixing_tank_v_dot_out_k = mixing_tank_v_dot_out[k]
    mixer_conc_in_1 = c_1[k]
    mixer_conc_in_2 = c_2[k]  # np.full(nT, c_nom[1])

    # Get concentration measurement
    # Note: we don't need control action to compute this
    # assume ratio = 50:50
    inputs = cas.DM([
        0.5 * mixing_tank_v_dot_out_k,
        mixer_conc_in_1,
        0.5 * mixing_tank_v_dot_out_k,
        mixer_conc_in_2,
        mixing_tank_v_dot_out_k
    ])
    yk = model.H(tk, xk, inputs, *params.values())
    idx = mixer_tank_system_dt.output_names.index("mixing_tank_conc_out")
    mixing_tank_conc_out = float(yk[idx])

    # Add measurement noise
    mixing_tank_conc_out_m = mixing_tank_conc_out + meas_error[k]

    # Output error
    error = tank_conc_out_sp - mixing_tank_conc_out_m

    # Compute ratio controller command
    u = float(ctrl(0.0, error, Tx=Ts))
    u = float(np.clip(u, umin, umax))

    # Turn off
    #u = 0.0

    # For debugging
    # print(
    #     f"k: {k}, y: {mixing_tank_conc_out_m:6.2f}, "
    #     f"r: {tank_conc_out_sp:6.2f}, e: {ek:6.2f}, "
    #     f"int(e): {integral_error:6.2f}, u: {u:6.3f}"
    # )

    # Implement control action
    R = R_nop - u

    # Manipulated inputs (calculated)
    mixer_v_dot_in_1 = mixing_tank_v_dot_out_k / (R + 1)
    mixer_v_dot_in_2 = R * mixer_v_dot_in_1

    inputs = cas.DM([
        mixer_v_dot_in_1,
        mixer_conc_in_1,
        mixer_v_dot_in_2,
        mixer_conc_in_2,
        mixing_tank_v_dot_out_k
    ])
    U.append(cas.DM(inputs).T)

    # Recompute all outputs with new control action
    yk = model.H(tk, xk, inputs, *params.values())
    Y.append(yk.T)

    # Simulate system forward one step
    xkp1 = model.F(tk, xk, inputs, *params.values())
    X.append(xkp1.T)

    tk = tkp1
    xk = xkp1

    # if k == 0:
    #     break

yk = model.H(tk, xk, inputs, *params.values())
Y.append(yk.T)
U = np.array(cas.vcat(U))
X = np.array(cas.vcat(X))
Y = np.array(cas.vcat(Y))

assert U.shape == (nT, mixer_tank_system_dt.nu)
assert X.shape == (nT + 1, mixer_tank_system_dt.n)
assert Y.shape == (nT + 1, mixer_tank_system_dt.ny)

sim_results = pd.concat(
    [
        pd.DataFrame(t_eval, columns=["time"]),
        pd.DataFrame(U, columns=mixer_tank_system_dt.input_names),
        pd.DataFrame(Y, columns=mixer_tank_system_dt.output_names),
    ],
    axis=1,
)
sim_results = sim_results.set_index("time")
sim_results.index.name = "Time (mins)"
sim_results


In [ ]:
fig, axes = make_tsplots(
    sim_results,
    plot_info,
    units=units,
)

# Add target concentration line
tank_conc_out_sp = 50.0  # setpoint for concentration
axes[-1].axhline(
    tank_conc_out_sp, linestyle="--", color="C4", label="Setpoint"
)

plt.tight_layout()
filename = "blending_ratio_ctrl.png"
plt.savefig(plot_dir / filename, dpi=300)
plt.show()


In [ ]:
rmse_tracking = float(
    np.sqrt(
        np.mean((tank_conc_out_sp - sim_results["mixing_tank_conc_out"]) ** 2)
    )
)
rmse_tracking


## Ratio Control with Smoothing


In [ ]:
# Buffer tank dimensions
buffer_tank_H = 9   # Height [m]
buffer_tank_A = 10  # Cross-sectional area [m^2]
buffer_tank_volume = buffer_tank_A * buffer_tank_H  # m^3

buffer_tank_D = np.sqrt(4 * buffer_tank_A / np.pi)  # Diameter [m]
flow_rate = 1.0  # m³/min
tau_h = buffer_tank_volume / flow_rate

print(f"Tank diameter: {buffer_tank_D:.2f} m")
print(f"Tank volume: {buffer_tank_volume:.2f} m³")
print(f"Tank residence time: {tau_h:.2f} min")

buffer_tank_model = MixingTankModelCT(D=buffer_tank_D, name="buffer_tank")
buffer_tank_model


In [ ]:
# Construct combined system of mixer, mixing_tank, and buffer tank
connections_combined = {
    "mixing_tank_v_dot_in": "mixer_v_dot_out",
    "mixing_tank_conc_in": "mixer_conc_out",
    "buffer_tank_v_dot_in": "mixing_tank_v_dot_out",
    "buffer_tank_conc_in": "mixing_tank_conc_out",
}

combined_system = connect_systems(
    [mixer_model, mixing_tank_model, buffer_tank_model],
    connections_combined,
    StateSpaceModelCT,
    name="combined_system",
    verbose_names=True,
)

combined_system_dt = StateSpaceModelDTFromCTRK4(combined_system, dt)
print_sys_dimensions(combined_system_dt)


In [ ]:
# Initial conditions
mixing_tank_L0 = 2.0  # mixing tank starts full
mixing_tank_m0 = 500.0
buffer_tank_L0 = buffer_tank_H  # buffer tank starts full
buffer_tank_conc0 = 50.0
buffer_tank_m0 = buffer_tank_conc0 * buffer_tank_volume
x0 = np.array([mixing_tank_L0, mixing_tank_m0, buffer_tank_L0, buffer_tank_m0])

# Simulation output time vector
t_eval = Ts * np.arange(nT + 1)
assert t_eval.shape == (nT + 1, )

model = combined_system_dt

# Flow rate disturbance
mixing_tank_v_dot_out = np.full(nT+1, 1.0)
mixing_tank_v_dot_out[t_eval >= 300] = 0.75

# Measurement noise
sd_e = 0.0
meas_error = np.random.normal(0, sd_e, size=nT+1)

# No model parameters in this case
params = {}

# Controller parameters
K_p = 0.01
K_i = 0.0005
u_bounds = (-0.999, 0.999)

# Normal operating point of ratio: v_dot_in_2 / v_dot_in_1
R_nop = 1.0

# Target concentration setpoint
tank_conc_out_sp = 50.0

# Controller states
u = 0.0
integral_error = 0.0

def anti_windup(dui, windup_mode):
    if windup_mode == "both" or windup_mode == "lower":
        dui = max(dui, 0)
    if windup_mode == "both" or windup_mode == "upper":
        dui = min(dui, 0)
    return dui

# Simulation loop
U = []
X = [cas.DM(x0).T]
Y = []
xk = x0
tk = t_eval[0]
for k in range(nT):
    tkp1 = t_eval[k + 1]

    # Disturbance inputs
    mixing_tank_v_dot_out_k = mixing_tank_v_dot_out[k]
    buffer_tank_v_dot_out = mixing_tank_v_dot_out_k
    mixer_conc_in_1 = c_1[k]
    mixer_conc_in_2 = c_2[k]

    # Get concentration measurement assuming ratio = 50:50
    inputs = cas.DM([
        0.5 * mixing_tank_v_dot_out_k,
        mixer_conc_in_1,
        0.5 * mixing_tank_v_dot_out_k,
        mixer_conc_in_2,
        mixing_tank_v_dot_out_k,
        buffer_tank_v_dot_out,
    ])
    yk = model.H(tk, xk, inputs, *params.values())
    idx = combined_system_dt.output_names.index("mixing_tank_conc_out")
    mixing_tank_conc_out = float(yk[idx])

    mixing_tank_conc_out_m = mixing_tank_conc_out + meas_error[k]

    # Implement ratio controller (PI)
    error = tank_conc_out_sp - mixing_tank_conc_out_m
    dup = K_p * error
    dui = K_i * error * Ts

    windup_mode = "none"
    dui = anti_windup(dui, windup_mode)
    du = dup + dui
    u = u + du
    u = np.clip(u, u_bounds[0], u_bounds[1])

    # Implement control action
    R = R_nop + u

    # Manipulated inputs (calculated)
    mixer_v_dot_in_1 = mixing_tank_v_dot_out_k / (R + 1)
    mixer_v_dot_in_2 = R * mixer_v_dot_in_1

    inputs = cas.DM([
        mixer_v_dot_in_1,
        mixer_conc_in_1,
        mixer_v_dot_in_2,
        mixer_conc_in_2,
        mixing_tank_v_dot_out_k,
        buffer_tank_v_dot_out,
    ])
    U.append(cas.DM(inputs).T)

    # Recompute all outputs with new control action
    yk = model.H(tk, xk, inputs, *params.values())
    Y.append(yk.T)

    # Simulate system forward one step
    xkp1 = model.F(tk, xk, inputs, *params.values())
    X.append(xkp1.T)

    tk = tkp1
    xk = xkp1

yk = model.H(tk, xk, inputs, *params.values())
Y.append(yk.T)
U = np.array(cas.vcat(U))
X = np.array(cas.vcat(X))
Y = np.array(cas.vcat(Y))

assert U.shape == (nT, combined_system_dt.nu)
assert X.shape == (nT + 1, combined_system_dt.n)
assert Y.shape == (nT + 1, combined_system_dt.ny)

sim_results_ratio_ctrl_smoothing = pd.concat(
    [
        pd.DataFrame(t_eval, columns=["time"]),
        pd.DataFrame(U, columns=combined_system_dt.input_names),
        pd.DataFrame(Y, columns=combined_system_dt.output_names),
    ],
    axis=1,
)
sim_results_ratio_ctrl_smoothing = sim_results_ratio_ctrl_smoothing.set_index("time")
sim_results_ratio_ctrl_smoothing.index.name = "Time (mins)"
sim_results_ratio_ctrl_smoothing


In [ ]:
# Plot combined-system results including buffer-tank dynamics
units_combined = {
    **units,
    "buffer_tank_v_dot_out": flow_units,
    "buffer_tank_L": level_units,
    "buffer_tank_m": mass_units,
    "buffer_tank_conc_out": conc_units,
}

plot_info_combined = {
    "Flow Rates": {
        "In 1": {"var_name": "mixer_v_dot_in_1"},
        "In 2": {"var_name": "mixer_v_dot_in_2"},
        "Mixer out": {"var_name": "mixer_v_dot_out"},
        "Mixing Tank out": {"var_name": "mixing_tank_v_dot_out"},
        "Buffer out": {"var_name": "buffer_tank_v_dot_out"},
    },
    "Mixer Compositions": {
        "In 1": {"var_name": "mixer_conc_in_1", "color": "C0"},
        "In 2": {"var_name": "mixer_conc_in_2", "color": "C1"},
        "Out": {"var_name": "mixer_conc_out", "color": "C2", "ylim": [0, 100]},
    },
    "Mixing Tank Composition": {
        "Mixing Tank out": {
            "var_name": "mixing_tank_conc_out",
            "color": "C4",
            "ylim": [30, 70],
        },
    },
    "Buffer Tank Composition": {
        "Buffer Tank out": {
            "var_name": "buffer_tank_conc_out",
            "color": "C5",
            "ylim": [30, 70],
        },
    },
}

fig, axes = make_tsplots(
    sim_results_ratio_ctrl_smoothing,
    plot_info_combined,
    units=units_combined,
)

# Add target concentration line to mixing-tank and buffer composition plots
axes[-2].axhline(
    tank_conc_out_sp, linestyle="--", color="C4", label="Setpoint"
)
axes[-1].axhline(
    tank_conc_out_sp, linestyle="--", color="C5", label="Setpoint"
)

plt.tight_layout()
filename = "blending_ratio_ctrl_with_buffer_tank.png"
plt.savefig(plot_dir / filename, dpi=300)
plt.show()

rmse_tracking_mixing_tank = float(
    np.sqrt(
        np.mean(
            (
                tank_conc_out_sp
                - sim_results_ratio_ctrl_smoothing["mixing_tank_conc_out"]
            ) ** 2
        )
    )
)
rmse_tracking_buffer = float(
    np.sqrt(
        np.mean(
            (
                tank_conc_out_sp
                - sim_results_ratio_ctrl_smoothing["buffer_tank_conc_out"]
            ) ** 2
        )
    )
)
print(f"Mixing tank outlet RMSE: {rmse_tracking_mixing_tank:.4f}")
print(f"Buffer outlet RMSE:      {rmse_tracking_buffer:.4f}")

rmse_tracking_buffer


In [ ]:
# Additional summary figure using make_tsplots
plot_info_summary = {
    "Concentrations": {
        "In 1": {"var_name": "mixer_conc_in_1", "color": "C0"},
        "In 2": {"var_name": "mixer_conc_in_2", "color": "C1"},
        "Out": {"var_name": "buffer_tank_conc_out", "color": "C5"},
    }
}

fig, axes = make_tsplots(
    sim_results_ratio_ctrl_smoothing,
    plot_info_summary,
    units=units_combined,
    time_units="mins",
)

axes[0].axhline(tank_conc_out_sp, linestyle="--", color="gray")
axes[0].legend(loc="best")

plt.tight_layout()
filename = "blending_inlet_vs_outlet_comparison.png"
plt.savefig(plot_dir / filename, dpi=300)
plt.show()
